In [74]:
import numpy as np
import pandas as pd

# === Load and Prepare Data ===
df = pd.read_csv("/Users/jaytlinaskew/GitRepository/TimeSeries-Analysis/notebooks/observationEventForecasting/DataPreprocessing/FullIndicatorMatrix.csv")  # Using previously uploaded file
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(by=['indicator', 'date']).reset_index(drop=True)

# === Cutoff 7 Days Prior to April 30, 2025 ===
cutoff_date = pd.to_datetime("2025-04-30") - pd.Timedelta(days=7)
train_df = df[df['date'] <= cutoff_date]
future_df = df[df['date'] > cutoff_date]

# === Temporal Pattern Features ===
temporal_features = []

for indicator, group in df.groupby('indicator'):
    seen_series = group['seen'].values
    total_days = len(seen_series)
    total_seen = seen_series.sum()
    seen_ratio = total_seen / total_days
    seen_std = np.std(seen_series)

    # Calculate streaks
    streaks = []
    streak_val = seen_series[0]
    streak_len = 1

    for val in seen_series[1:]:
        if val == streak_val:
            streak_len += 1
        else:
            streaks.append((streak_val, streak_len))
            streak_val = val
            streak_len = 1
    streaks.append((streak_val, streak_len))

    seen_streaks = [l for v, l in streaks if v == 1]
    unseen_streaks = [l for v, l in streaks if v == 0]

    max_seen_streak = max(seen_streaks) if seen_streaks else 0
    max_unseen_streak = max(unseen_streaks) if unseen_streaks else 0

    # Inter-arrival stats
    seen_indices = np.where(seen_series == 1)[0]
    seen_intervals = np.diff(seen_indices)
    mean_seen_interval = np.mean(seen_intervals) if seen_intervals.size > 0 else np.nan
    std_seen_interval = np.std(seen_intervals) if seen_intervals.size > 0 else np.nan

    # Days since last seen
    last_seen_index = seen_indices[-1] if seen_indices.size > 0 else np.nan
    days_since_last_seen = total_days - last_seen_index - 1 if not np.isnan(last_seen_index) else np.nan

    temporal_features.append({
        'indicator': indicator,
        'total_days': total_days,
        'total_seen': total_seen,
        'seen_ratio': seen_ratio,
        'seen_std': seen_std,
        'max_seen_streak': max_seen_streak,
        'max_unseen_streak': max_unseen_streak,
        'mean_seen_interval': mean_seen_interval,
        'std_seen_interval': std_seen_interval,
        'days_since_last_seen': days_since_last_seen
    })

temporal_df = pd.DataFrame(temporal_features)

# === Activity-Visibility Mismatch Features ===
mismatch_features = []

for indicator, group in df.groupby('indicator'):
    activity = group['observations'].values
    seen = group['seen'].values

    high_thresh = np.percentile(activity, 75)
    low_thresh = np.percentile(activity, 25)

    high_activity_not_seen = np.logical_and(activity >= high_thresh, seen == 0)
    low_activity_seen = np.logical_and(activity <= low_thresh, seen == 1)

    mismatch_features.append({
        'indicator': indicator,
        'high_activity_not_seen_count': high_activity_not_seen.sum(),
        'high_activity_not_seen_ratio': high_activity_not_seen.sum() / len(activity),
        'low_activity_seen_count': low_activity_seen.sum(),
        'low_activity_seen_ratio': low_activity_seen.sum() / len(activity)
    })

mismatch_df = pd.DataFrame(mismatch_features)

# === Merge Temporal and Mismatch Features ===
features_df = pd.merge(temporal_df, mismatch_df, on='indicator', how='inner')
# add df seen column to merged_df
features_df['seen'] = df.groupby('indicator')['seen'].first().values
# === Correlation Analysis with 'seen' ===
# NOTE: This assumes you've added a final `seen` column per indicator (e.g., final day's status)
# If not, you may need to compute it before this section.
feature_columns = [col for col in features_df.columns if col not in ['indicator', 'seen']]
correlations = features_df[feature_columns + ['seen']].corr()['seen'].drop('seen').sort_values(key=abs, ascending=False)

# === Feature Engineering: Scoring Visibility ===
features_df['seen_score'] = (
    features_df['total_seen'] * 0.3 +
    features_df['seen_ratio'] * 50 +
    features_df['max_seen_streak'] * 0.2 -
    features_df['days_since_last_seen'] * 0.1
)

features_df['unseen_risk'] = (
    features_df['high_activity_not_seen_ratio'] * 100 +
    features_df['max_unseen_streak'] * 0.2 +
    features_df['days_since_last_seen'] * 0.5
)

# === Score Summary by Seen Status ===
score_summary = features_df.groupby('seen')[['seen_score', 'unseen_risk']].mean().reset_index()

# === Correlation Table Output ===
corr_df = correlations.reset_index()
corr_df.columns = ['Feature', 'Correlation with Seen']
corr_df = corr_df.sort_values(by='Correlation with Seen', key=abs, ascending=False)


In [16]:
features_df

,indicator,total_days,total_seen,seen_ratio,seen_std,max_seen_streak,max_unseen_streak,mean_seen_interval,std_seen_interval,days_since_last_seen,high_activity_not_seen_count,high_activity_not_seen_ratio,low_activity_seen_count,low_activity_seen_ratio,seen,seen_score,unseen_risk
0,102.129.153.158,119,2,0.016807,0.128547,1,65,21.000000,0.000000,32,117,0.983193,0,0.0,0,-1.559664,127.319328
1,102.129.153.43,119,3,0.025210,0.156763,1,51,36.000000,16.000000,4,116,0.974790,0,0.0,0,1.960504,109.678992
2,102.129.153.71,119,7,0.058824,0.235294,1,35,18.666667,11.556624,5,112,0.941176,0,0.0,0,4.741176,103.617647
3,102.165.16.161,119,6,0.050420,0.218810,1,69,6.000000,2.683282,19,113,0.949580,0,0.0,0,2.621008,118.257983
4,104.160.6.2,119,3,0.025210,0.156763,2,71,20.500000,19.500000,6,116,0.974790,0,0.0,0,1.960504,114.678992
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
201,international.standardbank.com/,119,1,0.008403,0.091284,1,98,NaN,NaN,98,118,0.991597,0,0.0,0,-8.879832,167.759664
202,pub.marq.com/,119,5,0.042017,0.200628,1,85,3.500000,1.658312,19,114,0.957983,0,0.0,0,1.900840,122.298319
203,realinvestmentadvice.com/,119,11,0.092437,0.289642,3,40,7.400000,7.337575,4,108,0.907563,0,0.0,0,8.121849,100.756303
204,www.emergencylighting.com/,119,3,0.025210,0.156763,2,76,7.000000,6.000000,76,116,0.974790,0,0.0,0,-5.039496,150.678992


In [75]:
acutals = df[(df['date'] == '2025-04-28') & (df['seen'] == 1)]
acutals

,API_UserName,date,indicator,observations,dayofweek,is_weekend,day,month,seen
1069,818860012482918321,2025-04-28,104.21.48.1,4003,0,False,28,4,1
1188,818860012482918321,2025-04-28,104.21.54.132,2,0,False,28,4,1
1307,818860012482918321,2025-04-28,104.21.61.32,3,0,False,28,4,1
2021,818860012482918321,2025-04-28,146.71.50.198,3,0,False,28,4,1
2854,818860012482918321,2025-04-28,156.146.63.130,3,0,False,28,4,1
2973,818860012482918321,2025-04-28,156.146.63.131,6,0,False,28,4,1
3806,818860012482918321,2025-04-28,156.146.63.166,1,0,False,28,4,1
3925,818860012482918321,2025-04-28,156.146.63.167,4,0,False,28,4,1
5353,818860012482918321,2025-04-28,156.146.63.179,2,0,False,28,4,1
6067,818860012482918321,2025-04-28,162.142.125.242,2,0,False,28,4,1


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("/Users/jaytlinaskew/GitRepository/TimeSeries-Analysis/notebooks/observationEventForecasting/DataPreprocessing/FullIndicatorMatrix.csv")  # Using previously uploaded file
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(by=['indicator', 'date']).reset_index(drop=True)

# Group and compute temporal features per day
df['cumulative_seen'] = df.groupby('indicator')['seen'].cumsum()

# Days since last seen (potential leakage via ffill)
df['days_since_last_seen'] = (
    df.groupby('indicator')['date']
    .transform(lambda x: x - x.where(df['seen'] == 1).ffill())
    .dt.days
)

# Seen streak: full cumulative running streak
def compute_seen_streak(series):
    streak = 0
    output = []
    for val in series:
        if val == 1:
            streak += 1
        else:
            streak = 0
        output.append(streak)
    return output

df['seen_streak'] = df.groupby('indicator')['seen'].transform(compute_seen_streak)

# Rolling mismatch features using large windows (leaky or too powerful)
df['rolling_seen_ratio_14d'] = df.groupby('indicator')['seen'].transform(lambda x: x.rolling(14, min_periods=1).mean())
df['rolling_seen_std_14d'] = df.groupby('indicator')['seen'].transform(lambda x: x.rolling(14, min_periods=1).std())

# Threshold-based mismatch
high_thresh = df.groupby('indicator')['observations'].transform(lambda x: x.rolling(14, min_periods=1).quantile(0.75))
low_thresh = df.groupby('indicator')['observations'].transform(lambda x: x.rolling(14, min_periods=1).quantile(0.25))

df['high_activity_not_seen'] = ((df['observations'] >= high_thresh) & (df['seen'] == 0)).astype(int)
df['low_activity_seen'] = ((df['observations'] <= low_thresh) & (df['seen'] == 1)).astype(int)

df['high_activity_not_seen_ratio_14d'] = df.groupby('indicator')['high_activity_not_seen'].transform(
    lambda x: x.rolling(14, min_periods=1).mean()
)
df['low_activity_seen_ratio_14d'] = df.groupby('indicator')['low_activity_seen'].transform(
    lambda x: x.rolling(14, min_periods=1).mean()
)


# Select features used for training
temporal_features_df = df[[
    'indicator', 'date', 'seen', 'observations',
    'cumulative_seen', 'days_since_last_seen', 'seen_streak',
    'rolling_seen_ratio_14d', 'rolling_seen_std_14d',
    'high_activity_not_seen_ratio_14d', 'low_activity_seen_ratio_14d'
]]


# Sort for proper rolling calculations
temporal_features_df = temporal_features_df.sort_values(['indicator', 'date']).reset_index(drop=True)

# Recalculate additional precision-improving features
temporal_features_df['rolling_seen_ratio_7d'] = (
    temporal_features_df.groupby('indicator')['seen'].transform(lambda x: x.rolling(7, min_periods=1).mean())
)

temporal_features_df['seen_ratio_diff'] = (
    temporal_features_df['rolling_seen_ratio_14d'] -
    temporal_features_df.groupby('indicator')['rolling_seen_ratio_14d'].shift(7)
)

temporal_features_df['rolling_obs_14d'] = (
    temporal_features_df.groupby('indicator')['observations'].transform(lambda x: x.rolling(14, min_periods=1).mean())
)

temporal_features_df['rolling_obs_7d'] = (
    temporal_features_df.groupby('indicator')['observations'].transform(lambda x: x.rolling(7, min_periods=1).mean())
)

temporal_features_df['obs_ratio_spike'] = (
    temporal_features_df['rolling_obs_14d'] / (temporal_features_df['rolling_obs_7d'] + 1e-6)
)

temporal_features_df['seen_streak_prev'] = (
    temporal_features_df.groupby('indicator')['seen_streak'].shift(1)
)

temporal_features_df['recent_drop_flag'] = (
    ((temporal_features_df['seen_streak_prev'] >= 5) & (temporal_features_df['seen_streak'] == 0)).astype(int)
)

temporal_features_df = temporal_features_df.fillna(0)


temporal_features_df.head()



,indicator,date,seen,observations,cumulative_seen,days_since_last_seen,seen_streak,rolling_seen_ratio_14d,rolling_seen_std_14d,high_activity_not_seen_ratio_14d,low_activity_seen_ratio_14d,rolling_seen_ratio_7d,seen_ratio_diff,rolling_obs_14d,rolling_obs_7d,obs_ratio_spike,seen_streak_prev,recent_drop_flag
0,102.129.153.158,2025-01-01,0,0,0,0.0,0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,102.129.153.158,2025-01-02,0,0,0,0.0,0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,102.129.153.158,2025-01-03,0,0,0,0.0,0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,102.129.153.158,2025-01-04,0,0,0,0.0,0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,102.129.153.158,2025-01-05,0,0,0,0.0,0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
5,102.129.153.158,2025-01-06,0,0,0,0.0,0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
6,102.129.153.158,2025-01-07,0,0,0,0.0,0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
7,102.129.153.158,2025-01-08,0,0,0,0.0,0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
8,102.129.153.158,2025-01-09,0,0,0,0.0,0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
9,102.129.153.158,2025-01-10,0,0,0,0.0,0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [78]:
# Tag indicators based on their historical predictability

# Step 1: Calculate key stats per indicator
predictability_stats = temporal_features_df.groupby('indicator').agg(
    total_days=('date', 'count'),
    total_seen=('seen', 'sum'),
    seen_days_std=('seen', 'std'),
    max_seen_streak=('seen_streak', 'max'),
    seen_rate=('seen', 'mean')
).reset_index()

# Step 2: Define criteria for "predictable"
# - Appears seen on at least 5 days
# - Seen rate ≥ 10%
# - Max seen streak ≥ 3
predictability_stats['is_predictable'] = (
    (predictability_stats['total_seen'] >= 5) &
    (predictability_stats['seen_rate'] >= 0.10) &
    (predictability_stats['max_seen_streak'] >= 3)
)

# Step 3: Merge tags back into full dataset
df_with_predictability = temporal_features_df.merge(
    predictability_stats[['indicator', 'is_predictable']],
    on='indicator',
    how='left'
)


df_with_predictability

,indicator,date,seen,observations,cumulative_seen,days_since_last_seen,seen_streak,rolling_seen_ratio_14d,rolling_seen_std_14d,high_activity_not_seen_ratio_14d,low_activity_seen_ratio_14d,rolling_seen_ratio_7d,seen_ratio_diff,rolling_obs_14d,rolling_obs_7d,obs_ratio_spike,seen_streak_prev,recent_drop_flag,is_predictable
0,102.129.153.158,2025-01-01,0,0,0,0.0,0,0.000000,0.000000,1.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0,False
1,102.129.153.158,2025-01-02,0,0,0,0.0,0,0.000000,0.000000,1.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0,False
2,102.129.153.158,2025-01-03,0,0,0,0.0,0,0.000000,0.000000,1.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0,False
3,102.129.153.158,2025-01-04,0,0,0,0.0,0,0.000000,0.000000,1.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0,False
4,102.129.153.158,2025-01-05,0,0,0,0.0,0,0.000000,0.000000,1.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24509,www.shorturl.at/,2025-04-25,1,5,50,0.0,1,0.357143,0.497245,0.0,0.0,0.428571,-0.142857,3.642857,6.571429,0.554348,0.0,0,True
24510,www.shorturl.at/,2025-04-26,1,2,51,0.0,2,0.428571,0.513553,0.0,0.0,0.571429,-0.071429,3.785714,6.857143,0.552083,1.0,0,True
24511,www.shorturl.at/,2025-04-27,0,0,51,1.0,0,0.428571,0.513553,0.0,0.0,0.571429,-0.071429,3.785714,6.857143,0.552083,2.0,0,True
24512,www.shorturl.at/,2025-04-28,1,1,52,0.0,1,0.428571,0.513553,0.0,0.0,0.714286,0.000000,3.642857,7.000000,0.520408,0.0,0,True


In [76]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier

# === Parameters ===
PREDICTION_DATE = pd.to_datetime("2025-04-28")
CUTOFF_DATE = PREDICTION_DATE - pd.Timedelta(days=7)

# === Final Feature Set ===
features = [
    'observations', 'cumulative_seen', 'days_since_last_seen', 'seen_streak',
    'rolling_seen_ratio_14d', 'rolling_seen_std_14d',
    'high_activity_not_seen_ratio_14d', 'low_activity_seen_ratio_14d',
    'seen_ratio_diff', 'obs_ratio_spike', 'recent_drop_flag'
]

# === Forecast Function ===
def forecast_seen_indicators(temporal_features_df, prediction_date, feature_cols, seen_col='seen'):
    cutoff_date = pd.to_datetime(prediction_date) - pd.Timedelta(days=7)

    train_df = temporal_features_df[temporal_features_df['date'] <= cutoff_date].copy()
    predict_df = train_df.groupby('indicator').tail(1).copy()

    # Stage 1: Rule filter
    filter_mask = (predict_df['seen_streak'] >= 3) & (predict_df['rolling_seen_ratio_14d'] >= 0.5)
    stage1_df = predict_df[filter_mask].copy()

    if stage1_df.empty:
        return []

    # Stage 2: Gradient Boosting Classifier
    model = GradientBoostingClassifier(n_estimators=100, random_state=42)
    X_train = train_df[feature_cols].fillna(0)
    y_train = train_df[seen_col]
    model.fit(X_train, y_train)

    X_stage2 = stage1_df[feature_cols].fillna(0)
    stage1_df['probability'] = model.predict_proba(X_stage2)[:, 1]
    stage1_df['predicted_seen'] = (stage1_df['probability'] >= 0.60).astype(int)

    return stage1_df.loc[stage1_df['predicted_seen'] == 1, 'indicator'].tolist()

# === Run the Forecast ===
predicted = forecast_seen_indicators(temporal_features_df, PREDICTION_DATE, features)

print(f"Forecasted indicators expected to be SEEN on {PREDICTION_DATE.date()}:")
for indicator in predicted:
    print("-", indicator)


Forecasted indicators expected to be SEEN on 2025-04-28:
- 104.21.48.1
- 104.21.54.132
- 162.142.125.247
- 162.142.125.255
- 34.160.111.145
- 68.67.179.164


In [ ]:
#save features_df to csv
#temporal_features_df.to_csv('/Users/jaytlinaskew/Documents/MainDocuments/Alyn/Datasets/temporal_features_df.csv', index=False)